# Phase 2 — Feature Engineering

# Set up libraries

In [1]:
import pandas as pd
import numpy as np

import datetime as dt
from datetime import date

from dateutil.relativedelta import relativedelta

# Loading the dataset

In [3]:
retail_data = pd.read_csv('../data/interim/retail_store_inventory_cleaned.csv')
display(retail_data.head())

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday/promotion,competitor_pricing,seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [4]:
retail_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                73100 non-null  str    
 1   store_id            73100 non-null  str    
 2   product_id          73100 non-null  str    
 3   category            73100 non-null  str    
 4   region              73100 non-null  str    
 5   inventory_level     73100 non-null  int64  
 6   units_sold          73100 non-null  int64  
 7   units_ordered       73100 non-null  int64  
 8   demand_forecast     73100 non-null  float64
 9   price               73100 non-null  float64
 10  discount            73100 non-null  int64  
 11  weather_condition   73100 non-null  str    
 12  holiday/promotion   73100 non-null  int64  
 13  competitor_pricing  73100 non-null  float64
 14  seasonality         73100 non-null  str    
dtypes: float64(3), int64(5), str(7)
memory usage: 8.4 MB


# Converting data type

In [5]:
retail_data['date'] = pd.to_datetime(retail_data['date'], errors= 'coerce')

In [6]:
retail_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   date                73100 non-null  datetime64[us]
 1   store_id            73100 non-null  str           
 2   product_id          73100 non-null  str           
 3   category            73100 non-null  str           
 4   region              73100 non-null  str           
 5   inventory_level     73100 non-null  int64         
 6   units_sold          73100 non-null  int64         
 7   units_ordered       73100 non-null  int64         
 8   demand_forecast     73100 non-null  float64       
 9   price               73100 non-null  float64       
 10  discount            73100 non-null  int64         
 11  weather_condition   73100 non-null  str           
 12  holiday/promotion   73100 non-null  int64         
 13  competitor_pricing  73100 non-null  float64       
 14  s

# DATE FEATURES

In [ ]:
# Extract year component
retail_data['year'] = retail_data['date'].dt.year

# Extract month component
retail_data['month'] = retail_data['date'].dt.month

# Extract quarter component
retail_data['quarter'] = retail_data['date'].dt.quarter

# Extract week of the year component
retail_data["week"] = retail_data["date"].dt.isocalendar().week.astype("Int64")

# Extract day in the week component where Monday = 1, Tuesday = 2,...., Sunday = 7
retail_data["day_of_week"] = retail_data["date"].dt.isocalendar().day

# DISCOUNT STANDARDIZATION

In [ ]:
# converting data into decimal format for easy calculations
retail_data['discount'] = np.where(retail_data['discount'] > 1, retail_data['discount'] / 100, retail_data['discount'])

# REVENUE FEATURES

In [ ]:
# Revenue before discount
retail_data["gross_revenue"] = (retail_data["units_sold"] * retail_data["price"])

# Revenue after discount
retail_data["net_revenue"] = (retail_data["units_sold"] * retail_data["net_price"]).round(2)

# Price after discount
retail_data["net_price"] = (retail_data["price"] * (1 - retail_data["discount"])).round(2)

# Monetary value of discount applied
retail_data["discount_amount"] = (retail_data["gross_revenue"] - retail_data["net_revenue"]).round(2)

# INVENTORY FEATURES

In [25]:
# Positive = inventory exceeds forecast 
# Negative = inventory below forecast
retail_data["inventory_gap"] = (retail_data["inventory_level"] - retail_data["demand_forecast"])


# Inventory available relative to forecast demand
# Avoid division by zero.
retail_data["inventory_demand_ratio"] = np.where( retail_data["demand_forecast"] > 0,
                                                retail_data["inventory_level"] / retail_data["demand_forecast"], np.nan)

# FORECAST PERFORMANCE FEATURES

In [26]:
# Positive value: Actual demand > Forecast
# Negative value: Actual demand < Forecast
retail_data["forecast_error"] = ( retail_data["units_sold"] - retail_data["demand_forecast"] )

# Percentage forecast error
# Avoid division by zero.
retail_data["forecast_error_pct"] = np.where( retail_data["units_sold"] != 0,
                                             (retail_data["forecast_error"] / retail_data["units_sold"]) * 100, np.nan)

# COMPETITIVE PRICING FEATURE

In [27]:
# Difference between our price and competitor price
retail_data["price_difference"] = (retail_data["price"] - retail_data["competitor_pricing"])

In [ ]:
# Viewing the final DataFrame
retail_data.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,...,day_of_week,gross_revenue,net_price,net_revenue,discount_amount,inventory_gap,inventory_demand_ratio,forecast_error,forecast_error_pct,price_difference
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,...,6,4254.50,26.80,3403.60,850.90,95.53,1.705175,-8.47,-6.669291,3.81
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,...,6,9451.50,50.41,7561.50,1890.00,59.96,1.416273,5.96,3.973333,-3.15
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,...,6,1819.35,25.19,1637.35,182.00,27.98,1.378006,-9.02,-13.876923,-3.33
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,...,6,1995.92,29.45,1796.45,199.47,406.82,7.542618,-1.18,-1.934426,-2.02
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,...,6,1030.96,73.64,1030.96,0.00,156.74,17.926566,4.74,33.857143,4.69


# Exporting the final dataset for analysis

In [30]:
retail_data.to_csv("../data/processed/retail_inventory_enriched.csv", index= False)